# Phase 2: Feature Mapping

Map Phase 1 patterns to existing engineered features. Identify **gaps** — patterns that appear
frequently in monster data but have no corresponding feature in the HP prediction model.

**Inputs:**
- `../data/trait_catalog.parquet`, `../data/action_catalog.parquet` (Phase 1)
- `../data/pattern_frequencies.csv` (Phase 1)
- `../../../../data/feature_contributions.csv` (main pipeline)

**Outputs:**
- `../data/feature_mapping.csv`
- `../data/unmapped_patterns.csv`

In [1]:
import pandas as pd
import numpy as np
import sys
from pathlib import Path
from collections import defaultdict

In [2]:
# Setup paths
cwd = Path.cwd()

if cwd.name == 'notebooks':
    LOCAL_DATA_DIR = '../data'
    MAIN_DATA_DIR = '../../../../data'
    HELPER_DIR = '../../../../notebooks/helper_files'
else:
    LOCAL_DATA_DIR = './notebooks/exploration/feature_classification/data'
    MAIN_DATA_DIR = './data'
    HELPER_DIR = './notebooks/helper_files'

# Import feature_config
sys.path.insert(0, str(Path(HELPER_DIR).resolve()))
from feature_config import PHASE2_FEATURES, get_phase3_features, CONDITIONS

# Load Phase 1 outputs
traits_df = pd.read_parquet(LOCAL_DATA_DIR + '/trait_catalog.parquet')
actions_df = pd.read_parquet(LOCAL_DATA_DIR + '/action_catalog.parquet')
pattern_freq_df = pd.read_csv(LOCAL_DATA_DIR + '/pattern_frequencies.csv')

# Load feature contributions from main pipeline
contributions_df = pd.read_csv(MAIN_DATA_DIR + '/feature_contributions.csv')

phase3_features = get_phase3_features()
all_model_features = PHASE2_FEATURES + phase3_features

print(f'Loaded {len(traits_df)} traits, {len(actions_df)} actions')
print(f'Pattern frequencies: {len(pattern_freq_df)} patterns')
print(f'Feature contributions: {len(contributions_df)} monsters')
print(f'\nModel features: {len(PHASE2_FEATURES)} Phase 2 + {len(phase3_features)} Phase 3 = {len(all_model_features)} total')

Loaded 521 traits, 882 actions
Pattern frequencies: 53 patterns
Feature contributions: 324 monsters

Model features: 9 Phase 2 + 38 Phase 3 = 47 total


## Section 2: Pattern-to-Feature Mapping

Mapping types:
- **direct** — Pattern maps 1:1 to a model feature
- **indirect** — Pattern is captured as part of a broader feature (e.g. DPR parsing)
- **unmapped** — No feature captures this pattern (GAP)

In [3]:
PATTERN_FEATURE_MAP = {
    # =====================================================================
    # DIRECT MAPPINGS - pattern maps 1:1 to a model feature
    # =====================================================================

    # Condition infliction (Phase 3 features)
    'inflicts_blinded':        {'features': ['inflicts_blinded'],        'mapping_type': 'direct', 'notes': 'Phase 3 condition feature'},
    'inflicts_charmed':        {'features': ['inflicts_charmed'],        'mapping_type': 'direct', 'notes': 'Phase 3 condition feature'},
    'inflicts_frightened':     {'features': ['inflicts_frightened'],     'mapping_type': 'direct', 'notes': 'Phase 3 condition feature'},
    'inflicts_grappled':       {'features': ['has_grapple'],            'mapping_type': 'direct', 'notes': 'Maps to has_grapple (Phase 3)'},
    'inflicts_incapacitated':  {'features': ['inflicts_incapacitated'], 'mapping_type': 'direct', 'notes': 'Phase 3 condition feature'},
    'inflicts_paralyzed':      {'features': ['inflicts_paralyzed'],     'mapping_type': 'direct', 'notes': 'Phase 3 condition feature'},
    'inflicts_petrified':      {'features': ['inflicts_petrified'],     'mapping_type': 'direct', 'notes': 'Phase 3 condition feature'},
    'inflicts_poisoned':       {'features': ['inflicts_poisoned'],      'mapping_type': 'direct', 'notes': 'Phase 3 condition feature'},
    'inflicts_prone':          {'features': ['inflicts_prone'],         'mapping_type': 'direct', 'notes': 'Phase 2 fixed penalty'},
    'inflicts_restrained':     {'features': ['inflicts_restrained'],    'mapping_type': 'direct', 'notes': 'Phase 3 condition feature'},
    'inflicts_stunned':        {'features': ['inflicts_stunned'],       'mapping_type': 'direct', 'notes': 'Phase 3 condition feature'},

    # Defensive abilities
    'magic_resistance':        {'features': ['has_magic_resistance_scaled'],     'mapping_type': 'direct', 'notes': 'Phase 3 scaled feature'},
    'legendary_resistance':    {'features': ['has_legendary_resistance_scaled'], 'mapping_type': 'direct', 'notes': 'Phase 3 scaled feature'},
    'regeneration':            {'features': ['has_regeneration_scaled'],         'mapping_type': 'direct', 'notes': 'Phase 3 scaled feature'},

    # Attack modifiers (Phase 2)
    'advantage_on_attack':     {'features': ['has_advantage_condition'],    'mapping_type': 'direct', 'notes': 'Phase 2 fixed penalty'},
    'pack_tactics':            {'features': ['has_advantage_condition'],    'mapping_type': 'direct', 'notes': 'Phase 2 fixed penalty (subset)'},

    # Spellcasting
    'spellcasting':            {'features': ['has_spellcasting', 'spellcaster_level'], 'mapping_type': 'direct', 'notes': 'Phase 3 features'},
    'innate_spellcasting':     {'features': ['has_spellcasting', 'spellcaster_level'], 'mapping_type': 'direct', 'notes': 'Phase 3 features'},

    # =====================================================================
    # DIRECT MAPPINGS - Core patterns from parsers.py
    # =====================================================================

    # has_advantage_condition() patterns
    'core_pack_tactics':           {'features': ['has_advantage_condition'], 'mapping_type': 'direct', 'notes': 'Parsed by has_advantage_condition()'},
    'core_blood_frenzy':           {'features': ['has_advantage_condition'], 'mapping_type': 'direct', 'notes': 'Parsed by has_advantage_condition()'},
    'core_reckless':               {'features': ['has_advantage_condition', 'has_attackers_advantage'], 'mapping_type': 'direct', 'notes': 'Also gives attackers advantage'},
    'core_ambusher':               {'features': ['has_advantage_condition'], 'mapping_type': 'direct', 'notes': 'Parsed by has_advantage_condition()'},
    'core_assassinate':            {'features': ['has_advantage_condition'], 'mapping_type': 'direct', 'notes': 'Parsed by has_advantage_condition()'},
    'core_grappler':               {'features': ['has_advantage_condition'], 'mapping_type': 'direct', 'notes': 'Parsed by has_advantage_condition()'},
    'core_has_advantage_attack':   {'features': ['has_advantage_condition'], 'mapping_type': 'direct', 'notes': 'Generic advantage on attack rolls'},
    'core_have_advantage_attack':  {'features': ['has_advantage_condition'], 'mapping_type': 'direct', 'notes': 'Plural variant'},
    'core_advantage_against':      {'features': ['has_advantage_condition'], 'mapping_type': 'direct', 'notes': 'Advantage on attack rolls against'},

    # has_disadvantage_condition() patterns
    'core_sunlight_sensitivity':   {'features': ['has_disadvantage_condition'], 'mapping_type': 'direct', 'notes': 'Parsed by has_disadvantage_condition()'},
    'core_sunlight_weakness':      {'features': ['has_disadvantage_condition'], 'mapping_type': 'direct', 'notes': 'Parsed by has_disadvantage_condition()'},
    'core_light_sensitivity':      {'features': ['has_disadvantage_condition'], 'mapping_type': 'direct', 'notes': 'Parsed by has_disadvantage_condition()'},

    # has_attackers_advantage() patterns
    'core_attackers_advantage_1':  {'features': ['has_attackers_advantage'], 'mapping_type': 'direct', 'notes': 'Parsed by has_attackers_advantage()'},
    'core_attackers_advantage_2':  {'features': ['has_attackers_advantage'], 'mapping_type': 'direct', 'notes': 'Parsed by has_attackers_advantage()'},

    # Legendary actions
    'core_legendary_actions':      {'features': ['legendary_action_count', 'legendary_actions_per_round'], 'mapping_type': 'direct', 'notes': 'Parsed by parse_legendary_actions()'},

    # Spellcaster level
    'core_spellcaster_level':      {'features': ['spellcaster_level', 'has_spellcasting'], 'mapping_type': 'direct', 'notes': 'Parsed by extract_spellcaster_level()'},
    'core_casts_as_level':         {'features': ['spellcaster_level', 'has_spellcasting'], 'mapping_type': 'direct', 'notes': 'Alternative spellcaster level pattern'},

    # =====================================================================
    # INDIRECT MAPPINGS - captured via DPR/stat parsing, not as own feature
    # =====================================================================

    # DPR pipeline
    'multiattack':             {'features': ['dpr_deviation'],    'mapping_type': 'indirect', 'notes': 'Captured by parse_dpr_from_json()'},
    'breath_weapon':           {'features': ['dpr_deviation'],    'mapping_type': 'indirect', 'notes': 'Damage captured by DPR parsing'},
    'extra_damage':            {'features': ['dpr_deviation'],    'mapping_type': 'indirect', 'notes': 'Conditional damage captured in DPR'},
    'recharge':                {'features': ['dpr_deviation'],    'mapping_type': 'indirect', 'notes': 'Recharge abilities parsed for DPR'},

    # Core charge/pounce patterns -> DPR
    'core_charge':             {'features': ['dpr_deviation'],    'mapping_type': 'indirect', 'notes': 'Parsed by parse_charge_bonus_attack()'},
    'core_pounce':             {'features': ['dpr_deviation'],    'mapping_type': 'indirect', 'notes': 'Parsed by parse_charge_bonus_attack()'},
    'core_rampage':            {'features': ['dpr_deviation'],    'mapping_type': 'indirect', 'notes': 'Parsed by parse_charge_bonus_attack()'},
    'core_trampling':          {'features': ['dpr_deviation'],    'mapping_type': 'indirect', 'notes': 'Parsed by parse_charge_bonus_attack()'},

    # Core multiattack count patterns -> DPR
    'core_multiattack_two':    {'features': ['dpr_deviation'],    'mapping_type': 'indirect', 'notes': 'Parsed by parse_dpr_from_json()'},
    'core_multiattack_three':  {'features': ['dpr_deviation'],    'mapping_type': 'indirect', 'notes': 'Parsed by parse_dpr_from_json()'},
    'core_multiattack_with':   {'features': ['dpr_deviation'],    'mapping_type': 'indirect', 'notes': 'Parsed by parse_dpr_from_json()'},

    # Core conditional damage -> DPR
    'core_taking_damage':      {'features': ['dpr_deviation'],    'mapping_type': 'indirect', 'notes': 'Conditional damage parsed in DPR'},
    'core_takes_damage':       {'features': ['dpr_deviation'],    'mapping_type': 'indirect', 'notes': 'Conditional damage parsed in DPR'},
    'core_plus_damage':        {'features': ['dpr_deviation'],    'mapping_type': 'indirect', 'notes': 'Conditional damage parsed in DPR'},

    # Saving throw patterns -> save DC
    'has_saving_throw':        {'features': ['save_dc_deviation'], 'mapping_type': 'indirect', 'notes': 'Save DC extracted by parse_save_dc()'},
    'save_or_effect':          {'features': ['save_dc_deviation'], 'mapping_type': 'indirect', 'notes': 'Indicates save-or-suck effect'},
    'half_damage_on_save':     {'features': ['save_dc_deviation'], 'mapping_type': 'indirect', 'notes': 'Half-damage save mechanic'},
    'repeat_save':             {'features': ['save_dc_deviation'], 'mapping_type': 'indirect', 'notes': 'Repeating save mechanic'},

    # Grapple / escape
    'escape_dc':               {'features': ['has_grapple'],      'mapping_type': 'indirect', 'notes': 'Escape DC implies grapple/restrain'},

    # Resistance/Immunity -> Phase 1.5 penalty
    'damage_resistance':       {'features': ['resist_immun_resistance_penalty'], 'mapping_type': 'indirect', 'notes': 'Captured by resist/immun penalty phase'},
    'damage_immunity':         {'features': ['resist_immun_immunity_penalty'],   'mapping_type': 'indirect', 'notes': 'Captured by resist/immun penalty phase'},
    'condition_immunity':      {'features': ['condition_immunity_count'],         'mapping_type': 'indirect', 'notes': 'Parsed from stat block, not description text'},

    # Core prone patterns -> inflicts_prone
    'core_knocked_prone':      {'features': ['inflicts_prone'],   'mapping_type': 'indirect', 'notes': 'Contributes to inflicts_prone detection'},
    'core_falls_prone':        {'features': ['inflicts_prone'],   'mapping_type': 'indirect', 'notes': 'Contributes to inflicts_prone detection'},

    # =====================================================================
    # UNMAPPED - No feature captures this pattern (GAPS)
    # =====================================================================

    # Shapechange / polymorph
    'shapechange':             {'features': [], 'mapping_type': 'unmapped', 'notes': 'No feature for shapechange ability'},
    'revert_form':             {'features': [], 'mapping_type': 'unmapped', 'notes': 'Paired with shapechange'},

    # HP manipulation
    'reduces_hp_max':          {'features': [], 'mapping_type': 'unmapped', 'notes': 'HP max reduction (vampires, wraiths)'},
    'regains_hp':              {'features': [], 'mapping_type': 'unmapped', 'notes': 'HP recovery abilities (beyond regeneration)'},
    'cant_regain_hp':          {'features': [], 'mapping_type': 'unmapped', 'notes': 'Prevents target HP recovery'},
    'heals_on_damage':         {'features': [], 'mapping_type': 'unmapped', 'notes': 'Heals equal to damage dealt'},

    # Movement / phasing
    'teleport':                {'features': [], 'mapping_type': 'unmapped', 'notes': 'Teleportation ability'},
    'incorporeal':             {'features': [], 'mapping_type': 'unmapped', 'notes': 'Move through objects/creatures'},
    'ethereal':                {'features': [], 'mapping_type': 'unmapped', 'notes': 'Ethereal plane access'},
    'phasing':                 {'features': [], 'mapping_type': 'unmapped', 'notes': 'Phase through solid matter'},

    # Death effects
    'death_burst':             {'features': [], 'mapping_type': 'unmapped', 'notes': 'Damage/effect on death'},
    'explodes_on_death':       {'features': [], 'mapping_type': 'unmapped', 'notes': 'Explosion on death'},

    # Special combat
    'swallow':                 {'features': [], 'mapping_type': 'unmapped', 'notes': 'Swallow/engulf mechanic'},
    'sneak_attack':            {'features': [], 'mapping_type': 'unmapped', 'notes': 'Sneak attack bonus damage'},
    'surprise_attack':         {'features': [], 'mapping_type': 'unmapped', 'notes': 'Surprise-based bonus'},
    'gaze_attack':             {'features': [], 'mapping_type': 'unmapped', 'notes': 'Gaze-based attacks (medusa, etc.)'},
    'life_drain':              {'features': [], 'mapping_type': 'unmapped', 'notes': 'Life drain mechanic'},

    # Control effects (beyond conditions)
    'charm_effect':            {'features': [], 'mapping_type': 'unmapped', 'notes': 'Charm effect beyond inflicts_charmed'},
    'fear_effect':             {'features': [], 'mapping_type': 'unmapped', 'notes': 'Frightful Presence / fear aura (beyond inflicts_frightened)'},
    'cant_move':               {'features': [], 'mapping_type': 'unmapped', 'notes': 'Speed reduced to 0'},
    'speed_reduction':         {'features': [], 'mapping_type': 'unmapped', 'notes': 'Speed reduction effect'},

    # Ongoing damage
    'ongoing_damage':          {'features': [], 'mapping_type': 'unmapped', 'notes': 'Recurring damage each turn'},
    'damage_on_start_turn':    {'features': [], 'mapping_type': 'unmapped', 'notes': 'Damage at start of turn'},
    'damage_on_end_turn':      {'features': [], 'mapping_type': 'unmapped', 'notes': 'Damage at end of turn'},

    # Stealth / hide
    'bonus_action_hide':       {'features': [], 'mapping_type': 'unmapped', 'notes': 'Hide as bonus action (Nimble Escape)'},
    'stealth_advantage':       {'features': [], 'mapping_type': 'unmapped', 'notes': 'Advantage on Stealth checks'},

    # Undead
    'undead_fortitude':        {'features': [], 'mapping_type': 'unmapped', 'notes': 'Drop to 0 HP -> CON save to survive'},

    # Misc
    'damage_reduction':        {'features': [], 'mapping_type': 'unmapped', 'notes': 'Flat damage reduction'},
    'regeneration_prevention': {'features': [], 'mapping_type': 'unmapped', 'notes': 'Conditions that stop regeneration'},
    'inflicts_unconscious':    {'features': [], 'mapping_type': 'unmapped', 'notes': 'Not in CONDITIONS list (10 occurrences)'},
    'aura':                    {'features': [], 'mapping_type': 'unmapped', 'notes': 'Broad aura pattern - partially captured by specific conditions'},
}

# Count by type
type_counts = defaultdict(int)
for v in PATTERN_FEATURE_MAP.values():
    type_counts[v['mapping_type']] += 1

print(f'Total patterns mapped: {len(PATTERN_FEATURE_MAP)}')
for mt, count in sorted(type_counts.items()):
    print(f'  {mt}: {count}')

Total patterns mapped: 90
  direct: 35
  indirect: 24
  unmapped: 31


## Section 3: Build Mapping DataFrame

In [4]:
# Convert mapping dict to DataFrame
mapping_rows = []
for pattern, info in PATTERN_FEATURE_MAP.items():
    mapping_rows.append({
        'pattern': pattern,
        'features': '|'.join(info['features']) if info['features'] else '',
        'mapping_type': info['mapping_type'],
        'notes': info['notes'],
    })

mapping_df = pd.DataFrame(mapping_rows)

# Join with pattern frequencies
mapping_df = mapping_df.merge(pattern_freq_df, on='pattern', how='left')
mapping_df['count'] = mapping_df['count'].fillna(0).astype(int)
mapping_df = mapping_df.sort_values(['mapping_type', 'count'], ascending=[True, False])

print(f'Mapping DataFrame: {len(mapping_df)} rows')
print(f'\nPatterns with occurrences: {(mapping_df["count"] > 0).sum()}')
print(f'Patterns with zero occurrences: {(mapping_df["count"] == 0).sum()}')
print(f'\nBy mapping type:')
for mt in ['direct', 'indirect', 'unmapped']:
    subset = mapping_df[mapping_df['mapping_type'] == mt]
    total_occ = subset['count'].sum()
    print(f'  {mt}: {len(subset)} patterns, {total_occ} total occurrences')

print(f'\nFull mapping table:')
print(mapping_df.to_string(index=False))

Mapping DataFrame: 90 rows

Patterns with occurrences: 53
Patterns with zero occurrences: 37

By mapping type:
  direct: 35 patterns, 342 total occurrences
  indirect: 24 patterns, 633 total occurrences
  unmapped: 31 patterns, 328 total occurrences

Full mapping table:
                   pattern                                           features mapping_type                                                          notes  count
         inflicts_grappled                                        has_grapple       direct                                  Maps to has_grapple (Phase 3)     37
            inflicts_prone                                     inflicts_prone       direct                                          Phase 2 fixed penalty     37
       inflicts_restrained                                inflicts_restrained       direct                                      Phase 3 condition feature     37
              spellcasting                 has_spellcasting|spellcaster_level       d

## Section 4: Gap Analysis

Unmapped patterns ranked by frequency — these are the most impactful gaps.

In [5]:
# Unmapped patterns sorted by frequency
gaps_df = mapping_df[mapping_df['mapping_type'] == 'unmapped'].copy()
gaps_df = gaps_df.sort_values('count', ascending=False)

print('=' * 70)
print('UNMAPPED PATTERNS (GAPS) - Ranked by Frequency')
print('=' * 70)
print(f'\nTotal gaps: {len(gaps_df)} patterns')
print(f'Total gap occurrences: {gaps_df["count"].sum()}')
print()

for _, row in gaps_df.iterrows():
    if row['count'] > 0:
        print(f"  {row['pattern']:30s}  count={row['count']:3d}  | {row['notes']}")

UNMAPPED PATTERNS (GAPS) - Ranked by Frequency

Total gaps: 31 patterns
Total gap occurrences: 328

  aura                            count= 63  | Broad aura pattern - partially captured by specific conditions
  fear_effect                     count= 45  | Frightful Presence / fear aura (beyond inflicts_frightened)
  shapechange                     count= 26  | No feature for shapechange ability
  regains_hp                      count= 25  | HP recovery abilities (beyond regeneration)
  revert_form                     count= 24  | Paired with shapechange
  cant_regain_hp                  count= 14  | Prevents target HP recovery
  ongoing_damage                  count= 14  | Recurring damage each turn
  swallow                         count= 13  | Swallow/engulf mechanic
  reduces_hp_max                  count= 12  | HP max reduction (vampires, wraiths)
  explodes_on_death               count= 11  | Explosion on death
  ethereal                        count= 10  | Ethereal plane access


In [6]:
# For each gap pattern, show which monsters have it
print('=' * 70)
print('MONSTERS BY GAP PATTERN')
print('=' * 70)

# Combine traits and actions for lookup
all_items = pd.concat([
    traits_df[['monster', 'patterns_str']],
    actions_df[['monster', 'patterns_str']]
])

# Only show top gaps (count >= 5)
top_gaps = gaps_df[gaps_df['count'] >= 5]

for _, row in top_gaps.iterrows():
    pattern = row['pattern']
    monsters_with = all_items[all_items['patterns_str'].str.contains(pattern, na=False)]['monster'].unique()
    print(f"\n### {pattern} ({row['count']} occurrences, {len(monsters_with)} monsters)")
    print(f"    {', '.join(sorted(monsters_with)[:15])}")
    if len(monsters_with) > 15:
        print(f"    ... and {len(monsters_with) - 15} more")

MONSTERS BY GAP PATTERN

### aura (63 occurrences, 55 monsters)
    Adult Black Dragon, Adult Brass Dragon, Adult Bronze Dragon, Adult Copper Dragon, Adult Gold Dragon, Adult Green Dragon, Adult Silver Dragon, Ancient Black Dragon, Ancient Blue Dragon, Ancient Brass Dragon, Ancient Bronze Dragon, Ancient Copper Dragon, Ancient Gold Dragon, Ancient Green Dragon, Ancient Red Dragon
    ... and 40 more

### fear_effect (45 occurrences, 24 monsters)
    Adult Black Dragon, Adult Blue Dragon, Adult Brass Dragon, Adult Bronze Dragon, Adult Copper Dragon, Adult Gold Dragon, Adult Green Dragon, Adult Red Dragon, Adult Silver Dragon, Adult White Dragon, Ancient Black Dragon, Ancient Blue Dragon, Ancient Brass Dragon, Ancient Bronze Dragon, Ancient Copper Dragon
    ... and 9 more

### shapechange (26 occurrences, 25 monsters)
    Adult Bronze Dragon, Adult Gold Dragon, Adult Silver Dragon, Ancient Brass Dragon, Ancient Bronze Dragon, Ancient Copper Dragon, Ancient Gold Dragon, Ancient Silver Dr

In [7]:
# Summary: coverage statistics
mapped_patterns = set(mapping_df[mapping_df['mapping_type'] != 'unmapped']['pattern'])
unmapped_patterns = set(mapping_df[mapping_df['mapping_type'] == 'unmapped']['pattern'])

def count_coverage(df, col='patterns_str'):
    """Count items with at least one mapped vs unmapped pattern."""
    has_mapped = 0
    has_unmapped = 0
    has_both = 0
    has_none = 0
    
    for patterns_str in df[col]:
        if pd.isna(patterns_str) or patterns_str == '':
            has_none += 1
            continue
        patterns = set(patterns_str.split('|'))
        is_mapped = bool(patterns & mapped_patterns)
        is_unmapped = bool(patterns & unmapped_patterns)
        
        if is_mapped and is_unmapped:
            has_both += 1
        elif is_mapped:
            has_mapped += 1
        elif is_unmapped:
            has_unmapped += 1
        else:
            has_none += 1
    
    return has_mapped, has_unmapped, has_both, has_none

print('=' * 70)
print('COVERAGE SUMMARY')
print('=' * 70)

for label, df in [('Traits', traits_df), ('Actions', actions_df)]:
    mapped, unmapped, both, none_ = count_coverage(df)
    total = len(df)
    print(f'\n{label} ({total} total):')
    print(f'  Mapped patterns only:   {mapped:4d} ({100*mapped/total:.1f}%)')
    print(f'  Unmapped patterns only:  {unmapped:4d} ({100*unmapped/total:.1f}%)')
    print(f'  Both mapped+unmapped:    {both:4d} ({100*both/total:.1f}%)')
    print(f'  No patterns detected:    {none_:4d} ({100*none_/total:.1f}%)')
    print(f'  Coverage rate:           {100*(mapped+both)/total:.1f}% have at least one mapped pattern')

COVERAGE SUMMARY

Traits (521 total):
  Mapped patterns only:    119 (22.8%)
  Unmapped patterns only:    59 (11.3%)
  Both mapped+unmapped:      31 (6.0%)
  No patterns detected:     312 (59.9%)
  Coverage rate:           28.8% have at least one mapped pattern

Actions (882 total):
  Mapped patterns only:    226 (25.6%)
  Unmapped patterns only:    25 (2.8%)
  Both mapped+unmapped:     105 (11.9%)
  No patterns detected:     526 (59.6%)
  Coverage rate:           37.5% have at least one mapped pattern


## Section 5: Coverage Analysis by Monster

Identify monsters with the most unmapped patterns and correlate with prediction error.

In [8]:
# For each monster, count mapped vs unmapped patterns
def get_monster_coverage(traits_df, actions_df):
    """Aggregate pattern coverage per monster."""
    monster_stats = defaultdict(lambda: {'mapped': set(), 'unmapped': set(), 'all_patterns': set()})
    
    for df in [traits_df, actions_df]:
        for _, row in df.iterrows():
            monster = row['monster']
            patterns_str = row.get('patterns_str', '')
            if pd.isna(patterns_str) or patterns_str == '':
                continue
            for p in patterns_str.split('|'):
                monster_stats[monster]['all_patterns'].add(p)
                if p in mapped_patterns:
                    monster_stats[monster]['mapped'].add(p)
                elif p in unmapped_patterns:
                    monster_stats[monster]['unmapped'].add(p)
    
    rows = []
    for monster, stats in monster_stats.items():
        rows.append({
            'monster': monster,
            'mapped_count': len(stats['mapped']),
            'unmapped_count': len(stats['unmapped']),
            'total_patterns': len(stats['all_patterns']),
            'unmapped_list': '|'.join(sorted(stats['unmapped'])),
        })
    
    return pd.DataFrame(rows)

monster_coverage = get_monster_coverage(traits_df, actions_df)
monster_coverage = monster_coverage.sort_values('unmapped_count', ascending=False)

print(f'Monsters with pattern data: {len(monster_coverage)}')
print(f'Monsters with unmapped patterns: {(monster_coverage["unmapped_count"] > 0).sum()}')
print(f'\nTop 30 monsters by unmapped pattern count:')
print(monster_coverage.head(30).to_string(index=False))

Monsters with pattern data: 255
Monsters with unmapped patterns: 130

Top 30 monsters by unmapped pattern count:
              monster  mapped_count  unmapped_count  total_patterns                                                                                                               unmapped_list
              Vampire             9               8              17 charm_effect|damage_on_start_turn|heals_on_damage|reduces_hp_max|regains_hp|regeneration_prevention|revert_form|shapechange
             Succubus             5               6              11                                                           aura|charm_effect|ethereal|reduces_hp_max|revert_form|shapechange
   Succubus (Incubus)             5               6              11                                                           aura|charm_effect|ethereal|reduces_hp_max|revert_form|shapechange
        Vampire Spawn             5               6              11                    damage_on_start_turn|explodes_on

In [9]:
# Cross-reference with prediction errors
error_df = contributions_df[['Name', 'actual_hp', 'predicted_hp', 'hp_error', 'hp_error_pct']].copy()
error_df = error_df.rename(columns={'Name': 'monster'})

monster_analysis = monster_coverage.merge(error_df, on='monster', how='inner')
monster_analysis['abs_error_pct'] = monster_analysis['hp_error_pct'].abs()

# Compare error rates: monsters with unmapped patterns vs without
has_unmapped = monster_analysis[monster_analysis['unmapped_count'] > 0]
no_unmapped = monster_analysis[monster_analysis['unmapped_count'] == 0]

print('=' * 70)
print('PREDICTION ERROR vs UNMAPPED PATTERNS')
print('=' * 70)

print(f'\nMonsters WITH unmapped patterns ({len(has_unmapped)}):')
print(f'  Mean |error %|:  {has_unmapped["abs_error_pct"].mean():.1f}%')
print(f'  Median |error %|: {has_unmapped["abs_error_pct"].median():.1f}%')

print(f'\nMonsters WITHOUT unmapped patterns ({len(no_unmapped)}):')
print(f'  Mean |error %|:  {no_unmapped["abs_error_pct"].mean():.1f}%')
print(f'  Median |error %|: {no_unmapped["abs_error_pct"].median():.1f}%')

# Show worst-predicted monsters with unmapped patterns
print(f'\nTop 20 worst-predicted monsters with unmapped patterns:')
worst = has_unmapped.sort_values('abs_error_pct', ascending=False).head(20)
print(worst[['monster', 'actual_hp', 'predicted_hp', 'hp_error', 'abs_error_pct',
             'unmapped_count', 'unmapped_list']].to_string(index=False))

PREDICTION ERROR vs UNMAPPED PATTERNS

Monsters WITH unmapped patterns (130):
  Mean |error %|:  24.6%
  Median |error %|: 15.1%

Monsters WITHOUT unmapped patterns (125):
  Mean |error %|:  36.3%
  Median |error %|: 16.7%

Top 20 worst-predicted monsters with unmapped patterns:
                  monster  actual_hp  predicted_hp   hp_error  abs_error_pct  unmapped_count                                unmapped_list
                  Octopus          3     -7.331765 -10.331765     344.392157               1                            stealth_advantage
                   Sprite          2      8.707601   6.707601     335.380037               1                         inflicts_unconscious
                   Quasit          7     16.424319   9.424319     134.633130               2                      revert_form|shapechange
             Pseudodragon          7     14.861245   7.861245     112.303501               1                         inflicts_unconscious
             Phase Spider     

## Section 6: Save Outputs

In [10]:
# Save feature mapping
mapping_df.to_csv(LOCAL_DATA_DIR + '/feature_mapping.csv', index=False)
print(f'Saved feature_mapping.csv: {len(mapping_df)} rows')

# Save unmapped patterns with monster lists
unmapped_output = gaps_df[gaps_df['count'] > 0].copy()

# Add monster lists
monster_lists = []
all_items = pd.concat([
    traits_df[['monster', 'patterns_str']],
    actions_df[['monster', 'patterns_str']]
])
for _, row in unmapped_output.iterrows():
    pattern = row['pattern']
    monsters = sorted(all_items[all_items['patterns_str'].str.contains(pattern, na=False)]['monster'].unique())
    monster_lists.append('|'.join(monsters))
unmapped_output['monsters'] = monster_lists

unmapped_output.to_csv(LOCAL_DATA_DIR + '/unmapped_patterns.csv', index=False)
print(f'Saved unmapped_patterns.csv: {len(unmapped_output)} rows')

# Summary
print(f'\n{"=" * 70}')
print('PHASE 2 SUMMARY')
print(f'{"=" * 70}')
print(f'Total patterns in mapping: {len(mapping_df)}')
print(f'  Direct:   {len(mapping_df[mapping_df["mapping_type"]=="direct"])}')
print(f'  Indirect: {len(mapping_df[mapping_df["mapping_type"]=="indirect"])}')
print(f'  Unmapped: {len(mapping_df[mapping_df["mapping_type"]=="unmapped"])}')
print(f'\nGap patterns with occurrences: {len(unmapped_output)}')
print(f'Total gap occurrences: {unmapped_output["count"].sum()}')

Saved feature_mapping.csv: 90 rows
Saved unmapped_patterns.csv: 26 rows

PHASE 2 SUMMARY
Total patterns in mapping: 90
  Direct:   35
  Indirect: 24
  Unmapped: 31

Gap patterns with occurrences: 26
Total gap occurrences: 328
